# Explorating Traffic Accident Patterns with Unsupervised Learning
- Authors: Travis St Peter, Duong Le

## Introduction
Traffic accidents are complex and chaotic events. They aren't just caused by bad driving, they're influenced by a web of factors, from weather changes and poor visibility to road conditions and missing stop signs. Because there are so many moving parts, traffic accident data is a great place to try unsupervised learning.

Instead of using data to predict a specific outcome, unsupervised learning lets us step back and let the data speak for itself. We can explore the overall structure of the information to uncover hidden patterns that might be invisible when just looking at individual variables.

For this project, we are looking at a dataset were every row represetns a unique crash. To figure out what really going on, we're taking a look at:
- The Logistics: Location (Latitude / Longitude), state, and how long the accident back up traffic.

- The Environment: Temperature, humidity, visibility, wind speed, and precipitation.

- The Road Features: Whether the crash happened near a crossing, junction, stop sign, or traffic signal.

We wanted to know if unsupervised learning can reveal meaningful structures or natural groupings within traffic accident records. In order to find the answer to this we used a couple techniques:
- PCA & SVD: These tools help us cut through the noise, simplifying the data to summarize the most important patterns without losing the big picture.

- Matrix Completion: Real-world data is messy and often missing values. We used this technique to logically estimate and fill in the blanks.

- K-Means & Hierarchical Clustering: These grouping methods helped us automatically organize accidents with similar characteristics together, helping us to see how different algorithms interpret the chaos of the roads.

## Theoretical Background

### Principal Component Analysis
Principal Component Analysis (PCA) is our go-to method for simplifying datasets that have too many numerical variables. Instead of looking at every single variable separately, PCA mathematically squishes the data down into new, combined variables called principal components.

These components are ordered by importance: the first component explains the most variation (or information) in the data, the second explains the next most, and so on. In our traffic accident dataset, instead of juggling temperature, humidity, visibility, duration, and road features all at once, PCA creates a few master components that capture the main themes of a crash.

**Implementation & Tuning**

Before we can run PCA, we have to standardize the data. This is a crucial implementation step because our variables are measured on completely different scales—temperature is in degrees, precipitation is in inches, and duration is in minutes. Standardizing ensures that variables with larger numbers don't accidentally overpower the smaller ones.

When it comes to "tuning" a PCA model, the main hyperparameter we have to choose is the number of principal components to keep. We evaluate this using a Scree Plot, which graphs the proportion of variance explained by each component. We look for an "elbow" in the plot where the information drops off, allowing us to keep only the components that matter and throw away the statistical noise.

**Interpretation**

Once the model is tuned and evaluated, we interpret the results in two ways:
- Scores: These tell us where each specific accident falls on our new principal component map. Accidents with similar scores have very similar profiles.

- Loadings: These are the recipe of each component. By looking at how strongly each original variable contributes to a component, we can what that component actually represents in teh real world.

### Singular Value Decomposition

Singular Value Decomposition (SVD) is closely related to PCA. While PCA focuses on variance, SVD approaches the problem by breaking the entire dataset into three smaller matrices that describe it's main pattern.

Mathematically, SVD breaks down our original data matrix $(X)$ into the following equation:
$$ X = U \Sigma V^{*}$$

**Interpretation**

- $U$: This matrix describes our accidents in the new reduced dimensional space. Each row is a specific crash, and the values show how strongly that crash relates to the major patterns found in the overall data.

- $\Sigma$: This is a diagonal matrix of weights. It tells us the sheer importance or strength of each underlying pattern.

- $V^{*}$: Similar to PCA loadings, this tells us how our original variables combine to create these new patterns.

**Implmentation & Tuning**

Just like PCA, implementing SVD requires us to first standardize our data so that variables with large ranges don't overshadow variables with small ranges.

When we tune an SVD model, the main hyperparameter is $k$, which represents the number of singular values (patterns) we want to keep. If we keep them all, we would just be recreating our original, messy data. Tuning $k$ to a smaller number allows us to compress the data and keep only the most imporant signals and drop the noisy, minor variations.

**Evaluation**

We evaluated our SVD model by looking at the reconstruction error. This measures how much information is lost when we try to rebuild our original dataset using only our chosen $k$ components. If a few singular compnents explain most of the variation that shows as low error and suggests that the dataset can be easily summarized. On the other hand, if the variation is spread across a bunch of components, it suggests that the causes causes of accidents are complex and scattered.

### Matrix Completion
Real world data is not often perfect, whether a weather sensor is offline or a police report leaves line unfilled datasets often have missing values. Instead of removing an entire row of data just because it's missing one value, matrix completion allows us to fill in the blanks in a logical manner.

Matrix Completion relies on the fact that variables are not completely independent. Because things like precipitation, humidity, and visibility are mathematically interconnected, the algorithm learns these hidden relationships to make highly educated estimates for the missing numbers.

**Implementation & Tuning**

Implementing this algorithm needs a matrix that is missing entries. To theoretically test the validity of the algorithm on perfectly clean data, the standard practice is to artificially hide known values, forcing the model to blind-guess them based solely on the surrounding data.

To get the most accurate estimates, we tune the models assumed complexity, or rank. We ned to find the mathematical sweet spot: if we tune the rank too high, the model just memorizes statistical noise, but if we set it too low, it oversimplifies and misses potentially subtle relationships betwen the variables.

**Evaluation**

The model is evaluated by taking the algorithm's estimated numbers and comparing them directly back to the actual, hidden numbers. By calculating the difference, we get an overall error score that tells us how well the model learned the underlying structure of the data.



### K-Means Clustering
K-means clustering is an unsupervised learning algorithm that automatically groups complex data into distinct clusters of similar observations. The goal is pretty simple: try to maximize the similarities within each group, while making sure the other grouns themselves are as different from each other as possible.

The algorithm works by randomly dropping a set number of center points, called centroids, into the data. For our dataset, every traffic accident is assigned to it's nearest centroid, the algorithm recalculates the exact center of those newly formed groups, and moves the centroids to those new locations.

**Implementation & Tuning**

Since K-Means relies entirely on finding the mathematical distance between accidents, making sure the data is standardized before running the algorithm is extremely important. Skipping this would mean that variables wtih large scales would entirely overshadow variables with smaller scales.

The hyperparameter we manually choose is $k$, which represents the exact number of clusters the algorithm will search for.

**Evaluation**

To figure out if we chose the right number for $k$, we evaluate the model using Elbow Plots and Silhouette Scores. These tools mathematically score how tight and choesive the clusters are, which helps us objectively decide how many groups seem reasonsable for the data.

**Interpretation**

Once the algorithms finish grouping, we interpret the results by looking at the averages of the original variables within each cluster. By looking at a cluster's underlying signature, we can give it a real-world label. For example, we might look at the data in Cluster 1 and realize it represents "High-Severity, Low-Visibility Winter Highway Crashes," while Cluster 2 represents "Minor Sunny Intersection Fender-Benders."

### Hierarchical Clustering
Hierarchical clustering is an unsupervised learning algorithm that organizes data into a massive, visual family tree called a dendrogram. Unlike K-means, which requires us to guess the number of clusters upfront, hierarchical clustering builds from the ground up. It starts by treating every single traffic accident as its own unique cluster, and then slowly pairs the most similar accidents together, step by step, until everything merges into one giant trunk.

**Implementation & Tuning**
Because this algorithm builds it's tree strictly by calculating the mathematical distance between data points, similar to K-means, standardizing the dataset beforehand is mandatory. Without standardization, a variable with large numbers would artificially skew the tree's structure.

Tuning this model requires two main choices. First, we choose where to cut the dendrogram to make the final number of clusters. Second, we choose a linkage method, which decides exactly how the distance between two growing clusters is calculated. The linkage method is significantly change the shape of the tree:
- Single linkage uses the shortest distance between points in two clusters.

- Complete linkage uses the largest distance between points in two clusters.

- Average linkage uses the average distance between points in two clusters.

- Ward linkage joins clusters in a way that minimizes the overall mathematical variance within the newly formed group.

**Evaluation**

We evaluate the model visually by inspecting the dendrogram. By comparing trees built with different linkage methods, we can see if our accident data has a highly stable, natural structure, or if the groupings are fragile and depend heavily on the math we used to build them.

**Interpretation**

We interpret the results by looking at the major branches of the tree. For example, the very first major split in the dendrogram might divide the data into Adverse Weather Crashes and Clear Weather Crashes. As we follow the branches further down, we can interpret the increasingly specific sub-groups like clear-weather highway crashes versus clear-weather intersection fender-benders.

## Methodology

### Dataset Overview
The dataset used in this project contains traffic accident records. Each row represents a single, unique accident, capturing a snapshot of the exact conditions at the time of the crash.

To explore the underlying causes and patterns, we analyzed a mix of variable types:
- Continuous Numerical Data: Latitude, longitude, accident distance, temperature, humidity, pressure, visibility, wind speed, and precipitation.

- Categorical & Boolean Data: State, weather condition, and binary indicators for whether the accident occurred near specific road features like a crossing, junction, stop sign, or traffic signal.

Unsupervised learning algorithms are fundamentally math equations, they strictly require numbers to calculate distances and find patterns. Therefore, our primary goal during data preparation was to translate this messy, mixed-type dataset into a clean, unified numerical matrix.

### Data Cleaning and Feature Preparation
The data originally came packed with a wide mix of information, including continuous numbers (temperature, humidity, pressure, visibility, wind speed), categorical text (states, weather descriptions), and raw timestamps. Because unsupervised learning algorithms are fundamentally math equations, they strictly require numerical data to calculate patterns.

To translate this messy, mixed-type dataset into a clean numerical matrix, we walked through a few specific preparation steps:
- Missing Values: First, we inspected the dataset for missing values. Fortunately, the original continuous columns were completely intact, meaning we didn't have to throw away any valuable rows right out of the gate.

- Feature Engineering (Time & Duration): Dates and times are easy for humans to read, but terrible for math equations. We converted the text-based `StartTime` and `EndTime` columns into a proper datetime format. By subtracting the start from the end, we created a  new numerical variable: `Duration` (the length of the accident in minutes). We also extracted broad time-based categories like `Year`, `Season`, and general `Time` of day to capture the temporal context of the crashes.

- Feature Selection (Trimming the Text): We made a deliberate choice to drop categorical text variables like `WeatherCondition`. It is possible to mathematically encode them, but doing so would have increased the size of the dataset and made the clusters harder to interpret. We also excluded the `State` variable, since algorithms rely on mathematical weight—even if we assign Washington the number 1 and Oregon the number 2, the algorithm will mistakenly assume Oregon is mathematically greater than Washington, which ruins the logic.

- Keeping the Booleans: While we dropped the text, we kept the Boolean road-feature variables (`Amenity`, `Crossing`, `Junction`, `Stop`, and `TrafficSignal`). Because these naturally translate to clean 0 (False) and 1 (True) indicators, they provided, algorithm-friendly context about the crash location.

### Standardization

Machine learning algorithms are completely blind to units of measurement; they only see raw numbers. In our dataset, an accident duration might be 120 (minutes), while the precipitation might be 0.5 (inches). To a human, 0.5 inches of rain is a major weather event. But to an algorithm, 120 is larger than 0.5, so it will mistakenly assume that duration is considerably more important than the rain simply because the number is bigger.

In order to make the numerical values more even, we used a tool called `StandardScaler. This algorithm mathematically transforms every column so that they all share a mean of 0 and a standard deviation of 1. Regardless of the original datas units, standardization removes them and forces every variable onto the exact same proportional scale. This ensures that a massive number like an accident duration doesn't accidentally drown out the subtle but important signals from smaller numbers like wind speed.

### Modeling Approach

#### PCA & SVD

With the dataset standardized, we were able to perform PCA. As discussed earlier, PCA's job is to summarize the chaos by creating new master variables called principal components. To figure out how much real-world information each of these new components actually captured, we calculated the proportion of variance explained.

Since looking at the raw numbers can be overwhelming or confusing, we visually tuned the model using Scree and Cumulative Variance Plots. These plots allow us to easily see the elbow, which is where teh component stops adding use information and instead starts memorizing statistical noise. Once we knew how many components to keep, we looked into the PCA loadings to interpret which original road and weather features were actually driving these major patterns.

Alongside PCA, we also ran the data through SVD. Because SVD is mathematically related to PCA, applying it gave us a little different view into the same underlying structures. By calculating the singular values, we could measure the precise amount of variation explained by each component, essentially double-checking and verifying the structural patterns we found in our PCA.

#### Matrix Completion
Even though our dataset from free of missing values, we still wanted to text matrix completion. To do this we forced the algorithm to mask a portion of the standardized numerical data, and created artificial blanks for the model to solve.

To fill these blanks in we used an imputation algorithm to estimate the hidden entries based on the surviving, observed data. However, we had to carefully tune the algorithm so that it didnt guess wildly. By adjusting it's rank to make sure it was learning the true relationships between variables instead of just memorizing random noise. Once the algorithm had made its guess, we compared the algorithm's imputed numbers against the original, hidden values in order to evaluate it.

By calculating the error between the guesses and the real numbers, we could interpret how well the model actually understood the dataset. A low error rate suggests that traffic variables are interconnected enough that we can reliably reconstruct the profile of a crash, even when a portion of the data is missing.

#### K-Means Clustering

K-means clustering was used to group accidents with similar numerical profiles. Since k-means requires the number of clusters to be chosen in advance, multiple values of $k$ were tested. To evaluate which $k$ was the best we used two visual tools: an Elbow Plot and Silhouette Scores. The elbow plot helped us track how tightly packed the groups were becoming, while the silhouette scores measured how cleanly separated the groups were from each other. By comparing these two metrics, we were able to find the mathematical sweet spot where the clusters were cohesive but distinct.

Once the best $k$ was chosen let used the model to sort the standardized data into clusters. The final step to help interpretation, was to take the algorithm's resulting cluster labels (Group 1, Group 2, Group 3, etc.) and attach them back onto our original, unstandardized dataset. This allowed us to translate the math back into the real world profiles that these groups represent.


#### Hierarchical Clustering

For our final unsupervised technique, we deployed Hierarchical Clustering. To ensure our results were robust and stable, we compared several linkage methods, including single, complete, average, and Ward linkage. Because each of these methods calculates the distance between clusters slightly differently, testing all of them allowed us to verify if our traffic accidents had a natural, undeniable structure, or if the groupings were just a fragile illusion created by a single formula.

To visually evaluate these groupings, we generated dendrograms using a smaller sample of observations. This was done to improve readability and to keep our visualizations clean.

The final clusters were interpreted by comparing the average values of the original variables within each specific group. This allowed us to identify what patterns of accidetns tended to be grouped together, adn how those groups differed from each other.

## Results

### PCA and SVD Results
**PCA**

The PCA resulted in new components summarizing the traffic accident dataset. The scree plot ans cumulative variance plot below show the proportion of variance explained by each principal component.

<center>
<table>
  <tr>
    <td><img src="https://raw.githubusercontent.com/TravisStPeterSU/practicalhomework4/0756bc7f7edc401074856b27a78a37bce4122efb/scree_plot.png" alt="Scree Plot"></td>
    <td><img src="https://raw.githubusercontent.com/TravisStPeterSU/practicalhomework4/0756bc7f7edc401074856b27a78a37bce4122efb/cumulative_variance.png" alt="Cumulative Variance"></td>
  </tr>
</table>
</center>


The first principal component accounted for 11.94% of the total variance, while the second component added another 9.16%. Expanding the scope, the first five components collectively captured 40.32% of the variance, and the top ten components reached a cumulative total of 61.81%."

<center>

| Principal Component | Explained Variance | Cumulative Variance |
| --------------- | --------------- | --------------- |
| PC1      | 0.1194 | 0.1194 |
| PC2      | 0.0916 | 0.2110 |
| PC3      | 0.0803 | 0.2913 |
| PC4      | 0.0583 | 0.3496 |
| PC5      | 0.0536 | 0.4032 |
| PC6      | 0.0466 | 0.4498 |
| PC7      | 0.0434 | 0.4932 |
| PC8      | 0.0430 | 0.5362 |
| PC9      | 0.0412 | 0.5774 |
| PC10      | 0.0407 | 0.6181 |

</center>

**SVD**

SVD was applied to the standardized traffic accident dataset as an alternative approach to dimensionality reduction. The SVD output produced a $U$ matrix with shape (2590, 27), a vector of singular values with shape (27, 27), and a $VT$ matrix with shape (27, 27).

The scree plot and cumulative variance plot below show the proportion of variance captured by each singular component.

<center>
<table>
  <tr>
    <td><img src="https://raw.githubusercontent.com/TravisStPeterSU/practicalhomework4/d9e5dd6f5aa06781e51a04a55d18d9c6111f9612/svd_scree.png" " alt="SVD Scree Plot"></td>
    <td><img src="https://raw.githubusercontent.com/TravisStPeterSU/practicalhomework4/d9e5dd6f5aa06781e51a04a55d18d9c6111f9612/svd_cumulative_variance.png"  alt="SVD Cumulative Variance"></td>
  </tr>
</table>
</center>

The results of our SVD were similar to those obtained through PCA, which is expected since PCA is computed using SVD. SVD scree plot also revealed two distinct elbows, one at 5 components and the second one at 20 components.

The first singular component accounted for 11.94% of the total variance, while the second component added another 9.16% for a total of 21.10%. Similarly to the PCA, the first five components collectively captureed 40.32% and the top ten reached a cumulative total of 61.81%.

<center>

| Principal Component | Explained Variance | Cumulative Variance |
| --------------- | --------------- | --------------- |
| SV1      | 0.1194 | 0.1194 |
| SV2      | 0.0916 | 0.2110 |
| SV3      | 0.0803 | 0.2913 |
| SV4      | 0.0583 | 0.3496 |
| SV5      | 0.0536 | 0.4032 |
| SV6      | 0.0466 | 0.4498 |
| SV7      | 0.0434 | 0.4932 |
| SV8      | 0.0430 | 0.5362 |
| SV9      | 0.0412 | 0.5774 |
| SV10      | 0.0407 | 0.6181 |

</center>

### Matrix Completion Results
To demonstrate matrix completion, 10% of the standardized numerical values were randomly masked and treated as missing. A low-rank SVD reconstruction was then used to estimate the hidden values. The matrix completetion process then compared 7021 masked values to their reconstructed values and obtained an reconstruction error, measured using RMSE of 0.7619.

<center>
  <img
    src="https://raw.githubusercontent.com/TravisStPeterSU/practicalhomework4/3d1087d725127a9e3f03addcce81d995af1c0c71/matrix_recon.png" alt="Matrix Reconstruction Error" width="60%">
</center>

### K-Means Clustering Results

K-means clustering was applied to the standardized traffic accident dataset. To evaluate the appropriate number of clusters $k$, both the within-cluster sum of squares and average silhouette scores were calculated across multiple values of $k$.

<center>
<table>
  <tr>
    <td><img src="https://raw.githubusercontent.com/TravisStPeterSU/practicalhomework4/1c4be0cea6a2c6a9c099447ee00fb13b2056770c/k_means_elbow.png" alt="K-Means Elbow Plot"width="100%">
    </td>
    <td><img src="https://raw.githubusercontent.com/TravisStPeterSU/practicalhomework4/1c4be0cea6a2c6a9c099447ee00fb13b2056770c/k_means_silhouette.png" alt="K-Means Silhouette Scores"width="100%"></td>
  </tr>
</table>
</center>

While the average silhouette score reached its maximum at $k = 6$, the final K-means model was fit using $k = 8$ clusters. The resulting cluster labels were appended to the dataset. To visualize the distribution of these 8 clusters, the accident records were plotted based on their coordinates from the first two principal components (PC1 and PC2).

<center>
  <img
    src="https://raw.githubusercontent.com/TravisStPeterSU/practicalhomework4/473c0b65803b50955302f867017eee2ac503817d/k_means_cluster.png"alt="K-Means Clusters in PCA Space" width="70%">
</center>



### Hierarchical Clustering Results

Hierarchical clustering was applied using four different linkage methods: single, complete, average, and Ward linkage. The dendrograms below show how accident records were joined together under each linkage method. In each dendrogram, the y-axis shows the linkage distance where clusters were merged.

<center>
<table>
  <tr>
    <td align="center"><img src="https://raw.githubusercontent.com/TravisStPeterSU/practicalhomework4/d57958977a85747c747eb158dc150fd3756acd73/single_link.png" alt="Single Linkage" width="100%"><br><em>Single linkage</em></td>
    <td align="center"><img src="https://raw.githubusercontent.com/TravisStPeterSU/practicalhomework4/d57958977a85747c747eb158dc150fd3756acd73/complete_link.png" alt="Complete Linkage" width="100%"><br><em>Complete linkage</em></td>
  </tr>
  <tr>
    <td align="center"><img src="https://raw.githubusercontent.com/TravisStPeterSU/practicalhomework4/d57958977a85747c747eb158dc150fd3756acd73/avg_link.png" alt="Average Linkage" width="100%"><br><em>Average linkage</em></td>
    <td align="center"><img src="https://raw.githubusercontent.com/TravisStPeterSU/practicalhomework4/d57958977a85747c747eb158dc150fd3756acd73/ward_link.png" alt="Ward Linkage" width="100%"><br><em>Ward linkage</em></td>
  </tr>
</table>
</center>

The silhouette scores for the four linkage methods are shown in the table below.

<center>

| Linkage Method | Silhouette Score |
|---|---:|
| Single | 0.6445 |
| Complete | 0.3672 |
| Average | 0.5423 |
| Ward | 0.1072 |

</center>

## Discussion


### Interpretation of PCA and SV

Our first goal was to reduce the dimensionality of this dataset to see if any hidden, global structures existed. Both PCA and SVD accomplished this, gave essentially the same story about the traffic accidents.

Plotting the the accidents onto the first two principal components (PC1 vs. PC2) gives us a math based birds eye view of the dataset. It is different than plotting two original features against each other. If we plotted `Temperature` against `Humidity`, we would only see a basic weather correlation. But by plotting PC1 against PC2, we are plotting the two greatest sources of mathematical variance in the entire dataset.

<center>
  <img src="https://raw.githubusercontent.com/TravisStPeterSU/practicalhomework4/caa4199c5aca1466b0feb4c5237cfb0d6ec58a0e/pc1_pc2.png"alt="Accidents Plotted in the First Two Principal Components" width="70%">
</center>

Our SVD analysis helped us interpret exactly what these two major axes represent:
- The $V^{*}$ Matrix: Describes how much our original variables contribute to the new dimensions. By looking at the loadings, we discovered that our dataset is driven by two distinct forces. The first dimension (PC1) is heavily driven by geography and roadway infrastructure. The second dimension (PC2) is driven by weather-related conditions (humidity, temperature, season, and time of day).

- The $U$ Matrix: The $U$ matrix contains the coordinates for each accident. If two accidents sit close to each other on our PC1 vs. PC2 scatter plot, it means their $U$ matrix values are nearly identical. In the real world, this means those two crashes share very similar patterns across weather conditions, roadway characteristics, and geography.

Additionally, our Scree plots showed a double elbow, suggesting that traffic accidents are impacted by a few large overarching patterns layered on more localized secondary patterns.

### Interpretation of Clusters

To try understand the patterns of these accidents, we used the K-means model $(k = 8)$. The silhouette scores suggested 6 cluters, 8 clusters gave a more granular, interpretable breakdown of the accident crash profiles. By looking at the underlying averages of the original features within each group,  patterns seemed to emerge:

- Intersection Incidents: One cluster was heavily defined by high averages for the `Crossing` and `TrafficSignal` indicators. This cluster isolated accidents that occurred at busy, signal-controlled intersections.

- Railway Proximity: Another cluster was characterized by a high average for the `Railway` indicator, grouping together crashes that happened near train tracks and crossings.

- Traffic Calming & Rare Features: Smaller, more specialized clusters  formed around less common road infrastructure, grouping together accidents that occurred near `Bump`, `TrafficCalming` zones, and `Roundabout`.

- Environmental & Duration Splits: Other than infrastructure, the remaining clusters were divided by environment and impact. For example, the algorithm  separated long-duration crashes from quick accidents, and further split those groups by weather isolating warm, dry crashes from those occurring in cold, highly humid conditions.

Ultimately, these differences suggest that our clusters are not just random mathematical noise; they appear to reflect specific, genuine accident profiles based on location, weather, road features, and duration.

### Limitations

The matrix completion experiment was an interesting test of how interconnected the variables within the dataset are. By intentionally masking 10% of our data, we forced the model to guess the missing values based solely on the surrounding context. Achieving an RMSE of 0.7619 suggests that while there is underlying structure, traffic accidents still contain a high degree of randomness. We can liekly estimate missing environmental data based on the time and location of a crash, but it likely wont be perfectly accurate.

While unsupervised learning allowed us to uncover the hidden environmental and infrastructural structures of traffic accidents, this analysis is not without limitations.

First, K-Means clustering assumes that clusters are relatively spherical and evenly sized, which may artificially split long, continuous patterns into arbitrary groups. Second, PCA and SVD are strictly linear models; if the relationship between weather and accident duration is highly non-linear, these algorithms will fail to capture the full details.

Finally, our dataset is strictly environmental and geographic. It does not include data on driver behavior, such as speeding, distracted driving, or driving under the influence. Because human error is such an important contributor to traffic accidents, and analysis based purely on roads and weather will inherently only tell half the story.

## Conclusions

Unsupervised machine learning can be a strong tool to gain insight into complex datasets. In this analysis, we took a dense collection of traffic accident records and used dimensionality reduction (PCA and SVD) to reveal that crashes are fundamentally governed by two primary forces: geographic infrastructure and environmental conditions. By applying clustering algorithms (K-Means and Hierarchical), we further refined individual accidents into recognizable crash profiles from minor intersection fender-benders to severe weather-related highway collisions.